# Notebook 2: The Data Lake

**Pattern:** Raw data dumped into object storage, queried with a lightweight engine. Schema-on-read.

**Stack:** MinIO (object storage) + DuckDB (query engine) on raw Parquet files

**Maple Trust Bank** — synthetic BFSI data

---
## Configuration — Plan A / Plan B

Toggle the `USE_PLAN_A` flag below to switch between:
- **Plan A:** watsonx.data lakehouse (IBM Cloud)
- **Plan B:** Local MinIO + DuckDB (runs anywhere with Docker)

In [ ]:
# ── Configuration ──────────────────────────────────────────────────────
USE_PLAN_A = False  # Set True for watsonx.data lakehouse; False for local MinIO + DuckDB

# Plan B defaults (local Docker)
MINIO_ENDPOINT = "localhost:9000"
MINIO_ACCESS_KEY = "minioadmin"
MINIO_SECRET_KEY = "minioadmin"
MINIO_BUCKET = "bfsi-raw"
MINIO_SECURE = False

# Data paths
DATA_DIR = "../data"
LINEAGE_PATH = f"{DATA_DIR}/lineage/lineage_graph.json"

if USE_PLAN_A:
    print("Plan A: watsonx.data lakehouse — configure COS credentials above.")
else:
    print("Plan B: Local MinIO + DuckDB — ensure 'docker compose up minio' is running.")

In [ ]:
import duckdb
import pandas as pd
import json
import os
import pyarrow as pa
import pyarrow.parquet as pq
import tempfile

pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 120)

---
## Section 1: The Pattern in One Paragraph

The **data lake** is what happens when you take the Hadoop-era mantra — "store first, ask questions later" — and move it to object storage. Every file, every format, every team dumps their data into a single bucket. There is no schema enforcement, no data quality gate, no catalog. The promise: cheap, infinitely scalable storage where data scientists can explore freely. The reality: after three years, nobody knows what's in there, who put it there, or whether it's still valid. The lake is the architectural equivalent of a junk drawer — useful if you're the one who filled it, bewildering to everyone else. Schema-on-read means you pay the cost of understanding your data every single time you query it, not once at ingestion.

---
## Section 2: When You'd Use It, When You Wouldn't

| Use when | Don't use when |
|----------|----------------|
| Cheap, scalable raw storage for heterogeneous data | You need ACID transactions or consistency guarantees |
| ML training sets that need raw, unprocessed features | Regulatory reporting requiring schema enforcement |
| Exploratory analytics and data science sandboxes | Governed, auditable data pipelines |
| Landing zone for batch ingestion from many sources | Users expect a curated, queryable data model |
| You have strong data engineering to curate downstream | You have no governance team or data stewards |
| Archive / cold storage for compliance retention | Sub-second query latency on structured data |

---
## Section 3: The Setup — Load Raw Parquet into MinIO, Query with DuckDB

In [ ]:
# Upload raw parquet files to MinIO (object storage)
from minio import Minio
from minio.error import S3Error

client = Minio(
    MINIO_ENDPOINT,
    access_key=MINIO_ACCESS_KEY,
    secret_key=MINIO_SECRET_KEY,
    secure=MINIO_SECURE,
)

# Create bucket if it doesn't exist
if not client.bucket_exists(MINIO_BUCKET):
    client.make_bucket(MINIO_BUCKET)
    print(f"Created bucket: {MINIO_BUCKET}")
else:
    print(f"Bucket already exists: {MINIO_BUCKET}")

In [ ]:
# Upload the four parquet files
parquet_files = ["branches.parquet", "customers.parquet", "accounts.parquet", "transactions.parquet"]

for filename in parquet_files:
    filepath = os.path.join(DATA_DIR, filename)
    if os.path.exists(filepath):
        client.fput_object(MINIO_BUCKET, filename, filepath)
        stat = client.stat_object(MINIO_BUCKET, filename)
        print(f"  Uploaded {filename} ({stat.size / 1024 / 1024:.1f} MB)")
    else:
        print(f"  WARNING: {filepath} not found — run 'python data/generate.py' first")

print(f"\nAll files in bucket '{MINIO_BUCKET}':")
for obj in client.list_objects(MINIO_BUCKET):
    print(f"  s3://{MINIO_BUCKET}/{obj.object_name}  ({obj.size / 1024 / 1024:.1f} MB)")

In [ ]:
# Connect DuckDB — query parquet files directly (schema-on-read)
con = duckdb.connect()

# For local Plan B, we query the parquet files directly from disk.
# In Plan A (watsonx.data), you'd use Presto/Spark connectors to COS.
print("DuckDB connected — querying raw parquet files (schema-on-read)")
print("No schema was defined. No data types were enforced. We just read what's there.")

In [ ]:
# Quick schema inspection — discover what's in the lake
for f in parquet_files:
    path = os.path.join(DATA_DIR, f)
    result = con.execute(f"DESCRIBE SELECT * FROM '{path}'").fetchdf()
    print(f"\n── {f} ──")
    print(result.to_string(index=False))

In [ ]:
# Row counts — what's in the lake?
print("📊 Reference Architecture Swimlane: Analytical Data Management & Storage")
print("   (Data Lake = 'Outside Software Hub' — raw object storage)\n")

for f in parquet_files:
    path = os.path.join(DATA_DIR, f)
    count = con.execute(f"SELECT COUNT(*) FROM '{path}'").fetchone()[0]
    print(f"  {f:30s} {count:>12,} rows")

---
## Section 4: Three Canonical Queries

### Q1: Total transaction volume by branch for Q3 2024

DuckDB on raw parquet — works fine, but note: no schema enforcement, no data quality checks.

In [ ]:
q1 = con.execute(f"""
    SELECT
        t.branch_id,
        b.name AS branch_name,
        b.region,
        COUNT(*)           AS txn_count,
        SUM(t.amount)      AS total_amount,
        AVG(t.amount)      AS avg_amount
    FROM '{DATA_DIR}/transactions.parquet' t
    JOIN '{DATA_DIR}/branches.parquet' b ON t.branch_id = b.branch_id
    WHERE t.timestamp >= '2024-07-01' AND t.timestamp < '2024-10-01'
    GROUP BY t.branch_id, b.name, b.region
    ORDER BY total_amount DESC
""").fetchdf()

print("Q1: Transaction volume by branch — Q3 2024 (top 10)")
print("Note: This query works, but nothing prevented bad data from being in these files.")
q1.head(10)

### Q2: Find all customers whose policy documents reference AML procedure X

The lake can't do this — same limitation as a warehouse, but worse. No structure, no search index, no catalog.

In [ ]:
print("Q2: Policy document search — AML procedure references")
print("="*60)
print()
print("RESULT: Cannot be answered.")
print()
print("The data lake stores raw files — parquet, CSV, JSON, PDF.")
print("There is no full-text search engine. No document index.")
print("No catalog to even FIND which files contain policy docs.")
print()
print("In a warehouse, at least you know the schema.")
print("In a lake, you don't even know what's in there.")
print()
print("→ This is the core limitation: lakes store bytes, not knowledge.")
print("→ Addressed in Notebook 6 (RAG) with vector search over documents.")

### Q3: Trace the lineage of branch_summary_quarterly back to source

The lake has zero lineage tracking built in. We load the lineage graph separately — it's external metadata.

In [ ]:
# Load the lineage graph — this is NOT part of the lake, it's external metadata
with open(LINEAGE_PATH) as f:
    lineage = json.load(f)

print("Q3: Lineage for 'consumed.branch_summary_quarterly'")
print("="*60)
print()
print(f"Lineage graph: {lineage['metadata']['total_nodes']} nodes, {lineage['metadata']['total_edges']} edges")
print()

# Trace backwards from the target
target = "consumed.branch_summary_quarterly"

def trace_lineage(target_node, edges, depth=0):
    """Recursively trace lineage backwards."""
    upstream = [e for e in edges if e["to"] == target_node]
    for edge in upstream:
        indent = "  " * depth
        print(f"{indent}← {edge['from']}")
        print(f"{indent}   transform: {edge['transform'][:60]}...")
        trace_lineage(edge["from"], edges, depth + 1)

print(f"Target: {target}")
trace_lineage(target, lineage["edges"])
print()
print("⚠️  This lineage graph is EXTERNAL to the lake.")
print("   The lake itself has zero knowledge of how data flows.")
print("   Someone had to build and maintain this graph separately.")

---
## Section 5: Where This Pattern Breaks — The Governance Gap

### Break 1: Anyone can write malformed data

No schema enforcement means corrupt data enters silently.

In [ ]:
# Create a parquet file with wrong types and missing fields
import pyarrow as pa
import pyarrow.parquet as pq

malformed_data = pa.table({
    "branch_id": [999, 888],                       # int instead of string!
    "name": ["Fake Branch", None],                  # null name
    "address": [None, None],                        # all nulls
    # 'region' is missing entirely
    # 'manager_id' is missing entirely
    "extra_junk_column": ["oops", "garbage"],       # extra column
})

malformed_path = os.path.join(DATA_DIR, "branches_malformed.parquet")
pq.write_table(malformed_data, malformed_path)
print("Wrote malformed parquet file to the lake.")
print("No gate stopped us. No schema validation. No alert.")

In [ ]:
# DuckDB reads it without complaint
result = con.execute(f"SELECT * FROM '{malformed_path}'").fetchdf()
print("DuckDB reads malformed data silently:")
print(result)
print()
print("⚠️  branch_id is integer (should be string like 'MTB-001')")
print("⚠️  name has a NULL")
print("⚠️  address is all NULLs")
print("⚠️  region and manager_id are completely missing")
print("⚠️  extra_junk_column was added — no one will know it doesn't belong")

In [ ]:
# What happens when you UNION the malformed file with the real one?
try:
    result = con.execute(f"""
        SELECT branch_id, name, address FROM '{DATA_DIR}/branches.parquet'
        UNION ALL
        SELECT branch_id, name, address FROM '{malformed_path}'
    """).fetchdf()
    print("Union succeeded — malformed data mixed with real data:")
    print(result.tail(5))
    print()
    print("Notice: branch_id types were silently coerced. This is how lakes become swamps.")
except Exception as e:
    print(f"Union failed with: {e}")
    print("Even DuckDB noticed the types don't match — but the lake didn't prevent the write.")

### Break 2: No ACID — partial writes leave orphaned data

In [ ]:
# Simulate a failed multi-file write — partial data left behind
import tempfile

# Suppose a pipeline writes 3 partition files, but crashes after 2
partial_dir = os.path.join(DATA_DIR, "_partial_write_demo")
os.makedirs(partial_dir, exist_ok=True)

# Write partition 1 and 2 successfully
for i in range(2):
    chunk = pa.table({
        "transaction_id": [f"TXN-PARTIAL-{i}-{j}" for j in range(100)],
        "amount": [float(j * 10) for j in range(100)],
        "status": ["complete"] * 100,
    })
    pq.write_table(chunk, os.path.join(partial_dir, f"part-{i}.parquet"))

# Partition 3 "fails" — simulate crash
print("Pipeline wrote 2 of 3 partition files, then crashed.")
print(f"Orphaned files in: {partial_dir}/")
print()

# Query the partial data — it looks like complete data
partial_result = con.execute(f"SELECT COUNT(*) as rows, SUM(amount) as total FROM '{partial_dir}/*.parquet'").fetchdf()
print("Query result on partial data (looks normal, but is INCOMPLETE):")
print(partial_result)
print()
print("⚠️  No transaction log. No rollback. No way to know this is partial.")
print("   A consumer will use this data assuming it's complete.")
print("   This is why lakes need Iceberg/Delta for ACID → see Notebook 3 (Lakehouse).")

In [ ]:
# Clean up demo artifacts
import shutil
if os.path.exists(malformed_path):
    os.remove(malformed_path)
if os.path.exists(partial_dir):
    shutil.rmtree(partial_dir)
print("Cleaned up demo files.")
print()
print('"The lake is where data goes to drown."')
print('"Without governance, it becomes a swamp."')

---
## Section 6: The IBM Stack Mapping

| Component | IBM Product | Swimlane |
|-----------|-------------|----------|
| Object Storage | IBM Cloud Object Storage (COS) | Analytical Data Management & Storage |
| Query Engine | watsonx.data (Presto, Spark) | Analytical Data Management & Storage |
| Ingestion | DataStage, Kafka (Event Streams) | Ingestion & Integration |
| Catalog (add-on) | IBM Knowledge Catalog | Discovery & Exploration |

**Swimlane:** The data lake sits in the "Outside Software Hub" box of the reference architecture — external object storage connected through the platform.

In [ ]:
print("📊 Reference Architecture Swimlane: Analytical Data Management & Storage")
print("   Sub-zone: 'Outside Software Hub' — external storage")
print()
print("IBM Product Mapping:")
print("  Object Storage  → IBM Cloud Object Storage (COS)")
print("  Query Engine    → watsonx.data (Presto / Spark)")
print("  Ingestion       → DataStage, IBM Event Streams (Kafka)")
print("  Catalog         → IBM Knowledge Catalog (governance add-on)")
print()
print("Note: The lake pattern is storage. IBM's value-add is the governance")
print("layer (Knowledge Catalog) that prevents it from becoming a swamp.")

---
## Section 7: BFSI Reality Check

Canadian banks universally have data lakes — usually on S3 or COS — used primarily for ML training data and data science sandboxes. The pattern works well for its intended purpose: cheap storage of raw, heterogeneous data that data scientists can explore freely. The problem appears around year three. The lake has grown to petabytes. No one knows what's in most of the buckets. The original team has moved on. Metadata is sparse or missing. Sensitive data (PII, account numbers) is scattered across files with no access controls. The compliance team can't certify what's in there for OSFI audits. One major bank discovered production ML models were training on data that had been "deleted" from the warehouse but still sat in the lake — a PIPEDA violation hiding in plain sight. Every bank's data lake is really a data swamp with a governance veneer.

In [ ]:
# Close DuckDB connection
con.close()
print("Notebook 2 complete.")
print()
print("Key takeaway: The data lake is cheap, scalable, and ungoverned.")
print("It works for raw storage but fails for anything requiring trust.")
print("Next: Notebook 3 (Lakehouse) adds ACID, schema evolution, and time travel.")